# Recombination with GRG

**Note:** To add sample nodes (e.g., offspring from recombination), use grgl from the `grg_modify_improve` branch:

```bash
git clone -b grg_modify_improve --recursive https://github.com/aprilweilab/grgl.git
cd grgl && python setup.py bdist_wheel && pip install --force-reinstall dist/*.whl
```

The `MutableGRG.set_samples(sample_nodes)` API lets you add new sample nodes by passing the full list of sample node IDs (including the new offspring).

In [3]:
import numpy as np

NEGATIVE_NODE_IDS = []  
def get_breakpoints(N, p=0.1, min_breakpoints=1):
    """
    Return breakpoint positions. Ensures at least min_breakpoints in interior
    so recombination is less likely to be entire interval from one parent.
    """
    bp = np.where(np.random.binomial(1, p, N))[0]
    # Ensure at least one interior breakpoint to avoid entire-interval inheritance
    interior = np.arange(1, N)
    while len(bp) < min_breakpoints and len(interior) >= min_breakpoints:
        bp = np.unique(np.concatenate([bp, np.random.choice(interior, min(min_breakpoints, len(interior)), replace=False)]))
        bp = np.sort(bp)
    return bp

def recombination_intervals(h1, h2, N):
    """
    Returns list of segments: [(source_parent_id, end_coord), ...]
    """
    bp = get_breakpoints(N)
    start = np.random.binomial(1, 0.5, 1)[0]
    parents = [h1, h2]
    
    segments = []
    for i, K in enumerate(bp):
        segments.append((parents[(start + i) % 2], K))
    segments.append((parents[(start + len(bp)) % 2], N))
    return segments

In [4]:
from pygrgl import MutableGRG, Mutation, save_grg, grg_to_cyto_json

def create_simple_grg():
    grg = MutableGRG(4, 1, True) 
    
    # Create 4 internal nodes (4, 5, 6, 7)
    grg.make_node()  # node 4 (has m6, m7)
    grg.make_node()  # node 5 (has m8)
    grg.make_node()  # node 6 (has m9)
    grg.make_node()  # node 7 (has m10, m11)
    
    # Add edges (parent -> child) 
    grg.connect(6, 0)  # 6 -> 0
    grg.connect(6, 5)  # 6 -> 5
    grg.connect(7, 5)  # 7 -> 5
    grg.connect(7, 4)  # 7 -> 4
    grg.connect(5, 1)  # 5 -> 1
    grg.connect(5, 4)  # 5 -> 4
    grg.connect(4, 1)  # 4 -> 1
    grg.connect(4, 2)  # 4 -> 2
    grg.connect(4, 3)  # 4 -> 3
    
    # Add mutations (m1-m11)
    grg.add_mutation(Mutation(0, "A", "G"), 1)   # m1 on node 1
    grg.add_mutation(Mutation(1, "C", "T"), 0)   # m2 on node 0
    grg.add_mutation(Mutation(2, "G", "A"), 0)   # m3 on node 0
    grg.add_mutation(Mutation(3, "T", "C"), 2)   # m4 on node 2
    grg.add_mutation(Mutation(4, "A", "T"), 3)   # m5 on node 3
    grg.add_mutation(Mutation(5, "C", "G"), 4)   # m6 on node 4
    grg.add_mutation(Mutation(6, "G", "C"), 4)   # m7 on node 4
    grg.add_mutation(Mutation(7, "T", "A"), 5)   # m8 on node 5
    grg.add_mutation(Mutation(8, "A", "C"), 6)   # m9 on node 6
    grg.add_mutation(Mutation(9, "C", "A"), 7)  # m10 on node 7
    grg.add_mutation(Mutation(10, "G", "T"), 7)  # m11 on node 7
    
    return grg

# Create and save the GRG
simple_grg = create_simple_grg()


print("=== Simple GRG Created ===")
print(f"Nodes: {simple_grg.num_nodes}")
print(f"Edges: {simple_grg.num_edges}")
print(f"Mutations: {simple_grg.num_mutations}")
print(f"Samples: {simple_grg.get_sample_nodes()}")
print()

# Display structure
print("Node details:")
for node_id in range(simple_grg.num_nodes):
    parents = simple_grg.get_up_edges(node_id)
    children = simple_grg.get_down_edges(node_id)
    muts = simple_grg.get_mutations_for_node(node_id)
    mut_names = [f"m{m+1}" for m in muts]  # Just show mutation names
    is_sample = "[SAMPLE]" if simple_grg.is_sample(node_id) else ""
    print(f"  Node {node_id}: parents={parents}, children={children}, muts={mut_names} {is_sample}")

# Save to file
save_grg(simple_grg, "simple_example.grg")
print()
print("Saved to simple_example.grg")

cyto_data = grg_to_cyto_json(simple_grg)
print()
print("Cytoscape JSON representation:")
print(f"  Nodes: {len(cyto_data['nodes'])}")
print(f"  Edges: {len(cyto_data['edges'])}")
for node in cyto_data['nodes']:
    print(f"    {node['data']}")
for edge in cyto_data['edges']:
    print(f"    {edge['data']}")

=== Simple GRG Created ===
Nodes: 8
Edges: 9
Mutations: 11
Samples: [0, 1, 2, 3]

Node details:
  Node 0: parents=[6], children=[], muts=['m2', 'm3'] [SAMPLE]
  Node 1: parents=[5, 4], children=[], muts=['m1'] [SAMPLE]
  Node 2: parents=[4], children=[], muts=['m4'] [SAMPLE]
  Node 3: parents=[4], children=[], muts=['m5'] [SAMPLE]
  Node 4: parents=[7, 5], children=[1, 2, 3], muts=['m6', 'm7'] 
  Node 5: parents=[6, 7], children=[1, 4], muts=['m8'] 
  Node 6: parents=[], children=[0, 5], muts=['m9'] 
  Node 7: parents=[], children=[5, 4], muts=['m10', 'm11'] 

Saved to simple_example.grg

Cytoscape JSON representation:
  Nodes: 8
  Edges: 9
    {'id': 'n0', 'label': 'id=0, S, mutations({1, 2})', 'is_sample': 'True'}
    {'id': 'n1', 'label': 'id=1, S, mutations({0})', 'is_sample': 'True'}
    {'id': 'n2', 'label': 'id=2, S, mutations({3})', 'is_sample': 'True'}
    {'id': 'n3', 'label': 'id=3, S, mutations({4})', 'is_sample': 'True'}
    {'id': 'n4', 'label': 'id=4, mutations({5, 6})',

In [5]:
GRG_FILE = "simple_example.grg" 

In [81]:
class NonDuplicationRecombination:
    """
    Non-duplication GRG recombination algorithm.
    
    Key operations:
    - Path Compression: Skip nodes with no relevant mutations
    - Pruning: Stop when ancestry is disjoint from query interval
    - Bubble Insertion: Split nodes when partial mutation inheritance is needed
    """

    debug_mode = False # Set to True to enable debug prints
    
    def __init__(self, grg):
        self.grg = grg
        self.genome_length = grg.bp_range[1]  # l_max
        
        self._mutation_cache = {}
        
    def _get_node_mutations(self, node_id):
        """Get mutation positions for a node (cached)."""
        if node_id not in self._mutation_cache:
            mut_ids = self.grg.get_mutations_for_node(node_id)
            mutations = []
            for mut_id in mut_ids:
                mut = self.grg.get_mutation_by_id(mut_id)
                mutations.append((mut_id, mut.position))
            self._mutation_cache[node_id] = mutations
        return self._mutation_cache[node_id]
    
    def _get_mutations_in_interval(self, node_id, L, R):
        """Get mutations on node_id that fall within [L, R)."""
        node_muts = self._get_node_mutations(node_id)
        return [(mut_id, pos) for mut_id, pos in node_muts if L <= pos < R]
    
    def _has_connected_descendant(self, node_id, connected, visited=None):
        """True if any descendant of node_id (via down edges) is in connected."""
        if visited is None:
            visited = set()
        if node_id in visited:
            return False
        visited.add(node_id)
        for child in self.grg.get_down_edges(node_id):
            if child in connected:
                return True
            if self._has_connected_descendant(child, connected, visited):
                return True
        return False

    def _has_connected_ancestor(self, node_id, connected, visited=None):
        """True if any ancestor of node_id (via up edges) is in connected."""
        if visited is None:
            visited = set()
        if node_id in visited:
            return False
        visited.add(node_id)
        for parent in self.grg.get_up_edges(node_id):
            if parent in connected:
                return True
            if self._has_connected_ancestor(parent, connected, visited):
                return True
        return False

    def _get_ancestral_coverage(self, node_id):
        """
        Compute the ancestral interval coverage Iu (mutations in ancestors ONLY).
        """
        # Entry point: only look at parents to exclude the current node's mutations
        parents = self.grg.get_up_edges(node_id)
        if not parents:
            return None
            
        positions = []
        visited = set() # Standard visited set to handle complex graph topologies
        
        for parent in parents:
            # Call the helper which includes the node it's called on
            parent_span = self._get_node_and_ancestor_span(parent, visited)
            if parent_span:
                positions.extend([parent_span[0], parent_span[1]])
        
        if not positions:
            return None
        return (min(positions), max(positions) + 1)

    def _get_node_and_ancestor_span(self, node_id, visited):
        """Recursive helper: gets span of mutations for this node and its ancestors."""
        if node_id in visited:
            return None
        visited.add(node_id)
        
        # Include this node's mutations
        node_muts = self._get_node_mutations(node_id)
        positions = [pos for _, pos in node_muts]
        
        # Recurse to its parents
        for parent in self.grg.get_up_edges(node_id):
            anc_span = self._get_node_and_ancestor_span(parent, visited)
            if anc_span:
                positions.extend([anc_span[0], anc_span[1]])
                
        if not positions:
            return None
        
        return (min(positions), max(positions))
    
    def _extract_bubble(self, node_id, relevant_mut_ids, offspring_id, interval):
        """
        Create a bubble node to split mutations.
    
        Creates a new node v that:
        - Contains the relevant mutations (moved from node_id)
        - Becomes a parent of node_id
        - Becomes a parent of the offspring
        
        Args:
            node_id: The node to split
            relevant_mut_ids: Mutation IDs to move to the bubble
            offspring_id: The offspring node to connect
            interval: The interval [L, R) being inherited
            
        Returns:
            The new bubble node ID
        """
        # Create new bubble node
        bubble_id = self.grg.make_node()
        
        # Move relevant mutations from node to bubble (Algorithm 3: v.M <- Mrel; u.M <- u.M \ Mrel)
        for mut_id in relevant_mut_ids:
            mut = self.grg.get_mutation_by_id(mut_id)
            # Add mutation to bubble node
            self.grg.add_mutation(mut, bubble_id)
            # Remove mutation from original node (required for proper node splitting)
            self.grg.remove_mutation(mut_id, node_id)
        
        # Connect bubble -> node_id (bubble is parent of original node)
        self.grg.connect(bubble_id, node_id)
        
        # Connect bubble -> offspring
        self.grg.connect(bubble_id, -offspring_id)
        
        # Invalidate cache for the affected nodes
        self._mutation_cache.pop(node_id, None)
        self._mutation_cache.pop(bubble_id, None)
        
        return bubble_id
    
    def _recurse_attach(self, node_id, offspring_id, L, R, visited=None, connected=None):
        """
        Recursively attach ancestry to offspring for interval [L, R).
        
        Algorithm 2: RecurseAttach
        
        Handles:
        - Full Coverage (I ⊆ Iu): Ancestors fully cover query
        - Disjoint (I ∩ Iu = ∅): Pruning - stop traversal
        - Partial Overlap: Decomposition - recurse on intersection
        
        Args:
            node_id: Current ancestor node
            offspring_id: The new offspring node
            L, R: Query interval [L, R)
            visited: Set of already visited nodes to avoid cycles
            connected: Set of nodes already connected to offspring (avoid duplicates)
        """
        if L >= R:
            return
        
        if visited is None:
            visited = set()
        if connected is None:
            connected = set()
        
        if node_id in visited:
            return
        visited.add(node_id)
        
        # Get mutations and coverage info
        all_muts = self._get_node_mutations(node_id)
        relevant_muts = self._get_mutations_in_interval(node_id, L, R)
        
        all_mut_ids = set(mut_id for mut_id, _ in all_muts)
        relevant_mut_ids = set(mut_id for mut_id, _ in relevant_muts)

        # Check mutation relevance
        has_all_relevant = (relevant_mut_ids == all_mut_ids) and len(all_mut_ids) > 0
        has_no_relevant = len(relevant_mut_ids) == 0
        has_partial_relevant = len(relevant_mut_ids) > 0 and relevant_mut_ids != all_mut_ids
        
        # Get ancestral coverage
        Iu = self._get_ancestral_coverage(node_id)
        
        if self.debug_mode:
            print(f"Visiting node {node_id}: all_muts={all_mut_ids}, relevant_muts={relevant_mut_ids}, Iu={Iu}")
        
        # Determine coverage status
        if Iu is None:
            if self.debug_mode:
                print("No ancestral mutations - treating as root node")
            if has_all_relevant:
                if self.debug_mode:
                    print("Root Node Case 1: All relevant mutations - connect directly")
                if (node_id not in connected and
                    not self._has_connected_descendant(node_id, connected) and
                    not self._has_connected_ancestor(node_id, connected)):
                    self.grg.connect(node_id, -offspring_id)
                    connected.add(node_id)

                    # Remove sample status if connecting an original sample node to avoid confusion with new offspring samples
                    current_samples = list(self.grg.get_sample_nodes())
                    if node_id in current_samples:
                        current_samples.remove(node_id)
                        self.grg.set_samples(current_samples)

                elif self.deug_mode:
                    print(f"Node {node_id} already connected or has connected relatives, skipping direct connection")
            
            if has_partial_relevant:
                if self.debug_mode:
                    print("Root Node Case 2: Partial relevant mutations - create bubble and connect")
                bubble_id = self._extract_bubble(node_id, list(relevant_mut_ids), 
                                                        offspring_id, (L, R))
                connected.add(bubble_id)
                # connected.add(node_id)
            
            return # Stop - lineage fully resolved

        
        # Check if disjoint (Pruning - Scenario 2)
        # I ∩ Iu = ∅
        ancestral_disjoint = R < Iu[0] or L > Iu[1]
        
        # Check if partial overlap between [L, R] and Iu
        ancestral_partial_overlap = (L <= Iu[0] <= R) or (L <= Iu[1] <= R)

        # Check interval coverage
        full_coverage = (Iu[0] >= L and Iu[1] <= R)  # Iu ⊆ I (node is fully consumed)
        
        if has_all_relevant and full_coverage:
            if self.debug_mode:
                print("Case 1: Full coverage with all relevant mutations - connect directly")
            if (node_id not in connected and
                not self._has_connected_descendant(node_id, connected) and
                not self._has_connected_ancestor(node_id, connected)):
                self.grg.connect(node_id, -offspring_id)
                connected.add(node_id)

                # Remove sample status if connecting an original sample node to avoid confusion with new offspring samples
                current_samples = list(self.grg.get_sample_nodes())
                if node_id in current_samples:
                    current_samples.remove(node_id)
                    self.grg.set_samples(current_samples)

            elif self.debug_mode:
                print(f"Node {node_id} already connected or has connected relatives, skipping direct connection")
            return  # Stop - lineage fully resolved
        
        if has_no_relevant and ancestral_disjoint:
            if self.debug_mode:
                print("Case 2: No relevant mutations and disjoint ancestry - prune")
            # End search no relevant mutations to be found
            return
        
        if has_no_relevant and not ancestral_disjoint:
            if self.debug_mode:
                print("Case 3: No relevant mutations but not disjoint ancestry - path compression")
            # Path Compression - bypass this node
            # Recurse on parents
            parents = self.grg.get_up_edges(node_id)
            
            # If there's partial overlap with ancestry, we need to adjust the interval for the parents to the intersection of [L, R) and Iu
            if ancestral_partial_overlap:
                newL = max(L, Iu[0]) if Iu else L
                newR = min(R, Iu[1]) if Iu else R
                L, R = newL, newR

            for parent in parents:
                self._recurse_attach(parent, offspring_id, L, R, visited, connected)
            return
        
        if has_partial_relevant:
            if self.debug_mode:
                print("Case 4: Partial relevant mutations - need to create bubble and maybe recurse upwards")
            # Cases where we have relevant mutations (partial or full) but not full coverage - need to create bubble and maybe recurse upwards
            bubble_id = self._extract_bubble(node_id, list(relevant_mut_ids), 
                                                    offspring_id, (L, R))
            connected.add(bubble_id)
            # connected.add(node_id)  # Treat split node as covered to avoid connecting ancestors

            if not ancestral_disjoint:
                # If there's partial overlap with ancestry, we need to adjust the interval for the parents to the intersection of [L, R) and Iu
                if ancestral_partial_overlap:
                    newL = max(L, Iu[0]) if Iu else L
                    newR = min(R, Iu[1]) if Iu else R
                    L, R = newL, newR

                parents = self.grg.get_up_edges(node_id)
                for parent in parents:
                    self._recurse_attach(parent, offspring_id, L, R, visited, connected)
            
            return
        
        if has_all_relevant:
            if self.debug_mode:
                print("Case 5: All relevant mutations but not full coverage - need to create bubble and maybe recurse upwards")
            # Cases where we have relevant mutations (partial or full) but not full coverage - need to create bubble and maybe recurse upwards
            bubble_id = self._extract_bubble(node_id, list(relevant_mut_ids), 
                                                    offspring_id, (L, R))
            connected.add(bubble_id)
            # connected.add(node_id)  # Treat split node as covered to avoid connecting ancestors

            if ancestral_partial_overlap:
                # If there's partial overlap with ancestry, we need to adjust the interval for the parents to the intersection of [L, R) and Iu
                newL = max(L, Iu[0]) if Iu else L
                newR = min(R, Iu[1]) if Iu else R
                L, R = newL, newR
                parents = self.grg.get_up_edges(node_id)
                for parent in parents:
                    self._recurse_attach(parent, offspring_id, L, R, visited, connected)
            
            return
    
    def recombine(self, haplotype_A, haplotype_B, breakpoint):
        """
        Generate offspring through recombination.
        
        Creates a new offspring node inheriting:
        - [0, breakpoint) from haplotype_A
        - [breakpoint, genome_length) from haplotype_B
        
        Args:
            haplotype_A: Node ID of first parent
            haplotype_B: Node ID of second parent  
            breakpoint: Crossover position
            
        Returns:
            Node ID of the new offspring
        """
        # Create offspring node
        offspring_id = self.grg.make_node(negative=True)
        
        # Track connected nodes to avoid duplicate edges
        connected = set()
        
        # Inherit from haplotype_A for [0, breakpoint)
        self._recurse_attach(haplotype_A, offspring_id, 0, breakpoint, 
                            visited=None, connected=connected)
        
        # Inherit from haplotype_B for [breakpoint, genome_length)
        self._recurse_attach(haplotype_B, offspring_id, breakpoint, self.genome_length,
                            visited=None, connected=connected)
        
        if offspring_id not in NEGATIVE_NODE_IDS:
            NEGATIVE_NODE_IDS.append(offspring_id)
        return -(NEGATIVE_NODE_IDS.index(offspring_id) + 1)
    
    def recombine_multi(self, segments):
        """
        Generate offspring from multiple segments.
        
        Args:
            segments: List of (parent_node_id, interval_end) tuples
                     where intervals are implicit from previous end
                     
        Returns:
            Node ID of the new offspring
        """
        # Create offspring node
        offspring_id = self.grg.make_node(negative=True)
        
        # Track connected nodes to avoid duplicate edges
        
        
        # Process each segment
        start = 0
        for parent_id, end in segments:
            if end > start:
                if self.debug_mode:
                    print("BREAK")
                connected = set()
                self._recurse_attach(parent_id, offspring_id, start, end,
                                    visited=None, connected=connected)
            start = end
        
        if offspring_id not in NEGATIVE_NODE_IDS:
            NEGATIVE_NODE_IDS.append(offspring_id)
        return -(NEGATIVE_NODE_IDS.index(offspring_id) + 1)

In [7]:
import pygrgl
from pygrgl import load_mutable_grg, grg_to_cyto_json
import re

try:
    from pygrgl.display import grg_to_cyto, DAG_STYLE
    from ipycytoscape import CytoscapeWidget
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False
    DAG_STYLE = None
    CytoscapeWidget = None

if 'NEGATIVE_NODE_IDS' not in globals():
    NEGATIVE_NODE_IDS = []

def display_grg(grg, title="GRG", negative_node_ids=None):
    """Display GRG - uses widget if available, otherwise text."""
    neg_ids = negative_node_ids if negative_node_ids is not None else NEGATIVE_NODE_IDS
    cyto = grg_to_cyto_json(grg, start_from=grg.get_root_nodes())

    for n in cyto['nodes']:
        node_id = int(n['data']['id'][1:])
        if node_id in neg_ids:
            disp = -(neg_ids.index(node_id) + 1)
            n['data']['label'] = n['data']['label'].replace(f'id={node_id}', f'id={disp}', 1)
        
        # Record positions of mutations for this node to display in label
        positions = []
        for mut_id in grg.get_mutations_for_node(node_id):
            mut = grg.get_mutation_by_id(mut_id)
            positions.append(mut.position)

        pattern = r"mutations\(\{.*?\}\)"
        positions_str = ", ".join(str(n) for n in positions)
        # Build the replacement string using your new contents
        replacement = f"mutations({{{positions_str}}})"

        # Replace the whole mutations({ ... }) with the new one
        new_str = re.sub(pattern, replacement, n['data']['label'])
        n['data']['label'] = new_str
        
    if WIDGETS_AVAILABLE:
        try:
            widget = CytoscapeWidget()
            widget.graph.add_graph_from_json(cyto, directed=True)
            widget.set_style(DAG_STYLE)
            widget.set_layout(name="dagre")
            display(widget)
            return
        except Exception as e:
            print(f"Widget display failed: {e}")
    
    print(f"\n{'='*50}")
    print(f" {title}")
    print(f"{'='*50}")
    print(f"Nodes: {len(cyto['nodes'])}, Edges: {len(cyto['edges'])}")
    print("\nNodes:")
    for n in cyto['nodes']:
        d = n['data']
        marker = "[S]" if d.get('is_sample') == 'True' else "   "
        print(f"  {marker} {d['id']}: {d['label']}")
    print("\nEdges (parent → child):")
    for e in cyto['edges']:
        src, tgt = e['data']['source'], e['data']['target']
        try:
            tgt_id = int(tgt[1:])
            tgt_display = f'n{-(neg_ids.index(tgt_id) + 1)}' if tgt_id in neg_ids else tgt
        except ValueError:
            tgt_display = tgt
        print(f"      {src} → {tgt_display}")
    print(f"{'='*50}\n")

grg = create_simple_grg()

def print_grg_state(grg, title="GRG State"):
    """Helper to print GRG state."""
    print(f"=== {title} ===")
    print(f"Nodes: {grg.num_nodes}, Edges: {grg.num_edges}, Mutations: {grg.num_mutations}")
    print(f"Samples: {grg.get_sample_nodes()}")
    print(f"Genome range: {grg.bp_range}")
    print()
    all_nodes = pygrgl.get_topo_order(grg, pygrgl.TraversalDirection.DOWN, grg.get_root_nodes())
    for node_id in all_nodes:
        display_id = -(NEGATIVE_NODE_IDS.index(node_id) + 1) if node_id in NEGATIVE_NODE_IDS else node_id
        up = grg.get_up_edges(node_id)
        down = grg.get_down_edges(node_id)
        muts = grg.get_mutations_for_node(node_id)
        mut_names = [f"m{m+1}" for m in muts]  # Just show mutation names
        is_sample = " [SAMPLE]" if grg.is_sample(node_id) else ""
        print(f"  Node {display_id}: parents={up}, children={down}, muts={mut_names}{is_sample}")

# Show initial state
#print_grg_state(grg, "BEFORE Recombination")
display_grg(grg, "BEFORE Recombination")

CytoscapeWidget(cytoscape_layout={'name': 'dagre'}, cytoscape_style=[{'selector': 'node', 'style': {'font-fami…

In [8]:
recomb = NonDuplicationRecombination(grg)

haplotype_A = 2  
haplotype_B = 3  
breakpoint = 3 

print("=" * 60)
print("RECOMBINATION")
print("=" * 60)
print(f"Haplotype A: {haplotype_A}")
print(f"Haplotype B: {haplotype_B}")
print(f"Breakpoint: {breakpoint}")
print()
print(f"Offspring inherits [0, {breakpoint}) from haplotype {haplotype_A}")
print(f"Offspring inherits [{breakpoint}, {grg.bp_range[1]}) from haplotype {haplotype_B}")
print()

# Perform recombination
offspring_id = recomb.recombine(haplotype_A, haplotype_B, breakpoint)

try:
    raw_id = NEGATIVE_NODE_IDS[abs(offspring_id) - 1]
    current_samples = list(grg.get_sample_nodes())
    grg.set_samples(current_samples + [raw_id])
except AttributeError:
    pass

print(f"Created offspring node: {offspring_id}")
print()

# Show state after recombination
print_grg_state(grg, "AFTER Recombination")
display_grg(grg, "AFTER Recombination")

RECOMBINATION
Haplotype A: 2
Haplotype B: 3
Breakpoint: 3

Offspring inherits [0, 3) from haplotype 2
Offspring inherits [3, 11) from haplotype 3

Created offspring node: -1

=== AFTER Recombination ===
Nodes: 9, Edges: 10, Mutations: 11
Samples: [0, 1, 2, 3, 8]
Genome range: (0, 11)

  Node 7: parents=[], children=[5, 4], muts=['m10', 'm11']
  Node 6: parents=[], children=[0, 5], muts=['m9']
  Node 5: parents=[6, 7], children=[1, 4], muts=['m8']
  Node 4: parents=[7, 5], children=[1, 2, 3], muts=['m6', 'm7']
  Node 3: parents=[4], children=[8], muts=['m5'] [SAMPLE]
  Node -1: parents=[3], children=[], muts=[] [SAMPLE]
  Node 2: parents=[4], children=[], muts=['m4'] [SAMPLE]
  Node 1: parents=[5, 4], children=[], muts=['m1'] [SAMPLE]
  Node 0: parents=[6], children=[], muts=['m2', 'm3'] [SAMPLE]


CytoscapeWidget(cytoscape_layout={'name': 'dagre'}, cytoscape_style=[{'selector': 'node', 'style': {'font-fami…

In [9]:
def verify_offspring_mutations(grg, offspring_id, haplotype_A, haplotype_B, breakpoint):
    """
    Verify that offspring has correct mutations based on recombination.
    
    Expected mutations:
    - From haplotype_A: all ancestral mutations with position < breakpoint
    - From haplotype_B: all ancestral mutations with position >= breakpoint
    """
    def get_all_ancestral_mutations(node_id, visited=None):
        """Get all mutation IDs from node and its ancestors."""
        if visited is None:
            visited = set()
        if node_id in visited:
            return []
        visited.add(node_id)
        
        mutations = list(grg.get_mutations_for_node(node_id))
        
        for parent in grg.get_up_edges(node_id):
            mutations.extend(get_all_ancestral_mutations(parent, visited))
        
        return mutations
    
    def mut_name(mut_id):
        return f"m{mut_id + 1}"
    
    def get_position(mut_id):
        return grg.get_mutation_by_id(mut_id).position
    
    genome_length = grg.bp_range[1]
    
    # Get ancestral mutations for both haplotypes
    muts_A = get_all_ancestral_mutations(haplotype_A)
    muts_B = get_all_ancestral_mutations(haplotype_B)
    
    # Expected mutations for offspring
    expected_from_A = [m for m in muts_A if get_position(m) < breakpoint]
    expected_from_B = [m for m in muts_B if get_position(m) >= breakpoint]
    
    expected_muts = set(expected_from_A + expected_from_B)
    
    # Get actual offspring mutations (traversing ancestry)
    grg_node_id = NEGATIVE_NODE_IDS[abs(offspring_id) - 1] if offspring_id < 0 else offspring_id
    actual_muts = set(get_all_ancestral_mutations(grg_node_id))
    
    print(f"Haplotype A (node {haplotype_A}) mutations: {[mut_name(m) for m in muts_A]}")
    print(f"Haplotype B (node {haplotype_B}) mutations: {[mut_name(m) for m in muts_B]}")
    print()
    print(f"Expected from A (< m{breakpoint}): {[mut_name(m) for m in expected_from_A]}")
    print(f"Expected from B (>= m{breakpoint}): {[mut_name(m) for m in expected_from_B]}")
    print()
    print(f"Expected: {sorted([mut_name(m) for m in expected_muts])}")
    print(f"Actual:   {sorted([mut_name(m) for m in actual_muts])}")
    print()
    
    if expected_muts == actual_muts:
        print("✓ Offspring mutations are CORRECT!")
        return True
    else:
        missing = expected_muts - actual_muts
        extra = actual_muts - expected_muts
        if missing:
            print(f"✗ Missing: {[mut_name(m) for m in missing]}")
        if extra:
            print(f"✗ Extra: {[mut_name(m) for m in extra]}")
        return False

# Verify the recombination
print("=== Verification ===")
verify_offspring_mutations(grg, offspring_id, haplotype_A, haplotype_B, breakpoint)

=== Verification ===
Haplotype A (node 2) mutations: ['m4', 'm6', 'm7', 'm10', 'm11', 'm8', 'm9']
Haplotype B (node 3) mutations: ['m5', 'm6', 'm7', 'm10', 'm11', 'm8', 'm9']

Expected from A (< m3): []
Expected from B (>= m3): ['m5', 'm6', 'm7', 'm10', 'm11', 'm8', 'm9']

Expected: ['m10', 'm11', 'm5', 'm6', 'm7', 'm8', 'm9']
Actual:   ['m10', 'm11', 'm5', 'm6', 'm7', 'm8', 'm9']

✓ Offspring mutations are CORRECT!


True

In [85]:
def generate_offspring(grg, h1, h2, N=None):
    """
    Generate a recombined offspring from two parent haplotypes.
    
    Uses the recombination_intervals function to generate random
    crossover breakpoints, then applies non-duplication recombination.
    
    Args:
        grg: MutableGRG instance
        h1: First parent haplotype node ID
        h2: Second parent haplotype node ID
        N: Genome length (defaults to grg.bp_range[1])
        
    Returns:
        Tuple of (offspring_node_id, segments)
    """
    if N is None:
        N = grg.bp_range[1]
    
    # Get recombination segments
    segments = recombination_intervals(h1, h2, N)
    
    # Create recombination handler
    recomb = NonDuplicationRecombination(grg)
    
    # Perform recombination
    offspring_id = recomb.recombine_multi(segments)

    try:
        current_samples = list(grg.get_sample_nodes())
        raw_id = NEGATIVE_NODE_IDS[abs(offspring_id) - 1]
        grg.set_samples(current_samples + [raw_id])
    except AttributeError:
        pass  
    
    return offspring_id, segments



print("=== Random Recombination Example ===")
grg_test = create_simple_grg() 

print(f"Loading: {GRG_FILE}")
print(f"Genome length: {grg_test.bp_range[1]}")
print(f"Initial samples: {grg_test.get_sample_nodes()}")
print()

display_grg(grg_test, "BEFORE random recombination")

offspring, segs = generate_offspring(grg_test, h1=0, h2=3)
print()
print(f"Generated offspring node: {offspring}")
print(f"Samples (including offspring): {grg_test.get_sample_nodes()}")
print(f"Segments inherited: {segs}")
print()

print("Segment breakdown:")
start = 0
for parent, end in segs:
    print(f"  [{start}, {end}): from parent {parent}")
    start = end

# Show after state
display_grg(grg_test, "AFTER random recombination")


=== Random Recombination Example ===
Loading: simple_example.grg
Genome length: 11
Initial samples: [0, 1, 2, 3]



CytoscapeWidget(cytoscape_layout={'name': 'dagre'}, cytoscape_style=[{'selector': 'node', 'style': {'font-fami…


Generated offspring node: -1
Samples (including offspring): [0, 1, 2, 3, 8]
Segments inherited: [(3, 10), (0, 11)]

Segment breakdown:
  [0, 10): from parent 3
  [10, 11): from parent 0


CytoscapeWidget(cytoscape_layout={'name': 'dagre'}, cytoscape_style=[{'selector': 'node', 'style': {'font-fami…

In [11]:
# Redo last generated recombination test case from above for a closer look

grg_test2 = create_simple_grg()

display_grg(grg_test2, "BEFORE multiple recombinations")

recomb = NonDuplicationRecombination(grg_test2)

recomb.recombine_multi(segs)

print("Segment breakdown:")
start = 0
for parent, end in segs:
    print(f"  [{start}, {end}): from parent {parent}")
    start = end

display_grg(grg_test2, "AFTER multiple recombinations")

CytoscapeWidget(cytoscape_layout={'name': 'dagre'}, cytoscape_style=[{'selector': 'node', 'style': {'font-fami…

Segment breakdown:
  [0, 1): from parent 0
  [1, 2): from parent 3
  [2, 7): from parent 0
  [7, 11): from parent 3


CytoscapeWidget(cytoscape_layout={'name': 'dagre'}, cytoscape_style=[{'selector': 'node', 'style': {'font-fami…